Generate augmented images

In [12]:
import os, cv2
from albumentations import (
    Compose, RandomResizedCrop, HorizontalFlip, VerticalFlip, Rotate,
    RandomBrightnessContrast, HueSaturationValue, GaussNoise,
    ShiftScaleRotate, OpticalDistortion, GridDistortion, ElasticTransform,
    CLAHE, RandomGamma, RGBShift, Sharpen, Emboss, RandomShadow,
    MultiplicativeNoise, ISONoise, ColorJitter
)
from tqdm import tqdm

# Input/output folders
src_dir = "Plum_Rust_Fungus"              # original images folder
out_dir = "Plum_Rust_Fungus_Augmented"    # output folder for augmented images
N_per_image = 25                               # augmented versions per image

os.makedirs(out_dir, exist_ok=True)

# ✅ Natural-looking augmentation pipeline (NO BLUR, NO BLACK BOXES)
aug = Compose([
    # Cropping & Resizing (minimal change)
    RandomResizedCrop(size=(256, 256), scale=(0.94, 1.0), ratio=(0.94, 1.06), p=0.6),
    
    # Geometric transformations (very subtle rotations)
    Rotate(limit=2, border_mode=cv2.BORDER_REFLECT, p=0.25),   # 2-degree rotation
    Rotate(limit=4, border_mode=cv2.BORDER_REFLECT, p=0.25),   # 4-degree rotation
    Rotate(limit=6, border_mode=cv2.BORDER_REFLECT, p=0.2),    # 6-degree rotation
    Rotate(limit=8, border_mode=cv2.BORDER_REFLECT, p=0.15),   # 8-degree rotation
    ShiftScaleRotate(shift_limit=0.04, scale_limit=0.04, rotate_limit=3, 
                     border_mode=cv2.BORDER_REFLECT, p=0.3),
    
    # Flips
    HorizontalFlip(p=0.5),
    VerticalFlip(p=0.2),
    
    # Elastic & Optical distortions (simulate leaf texture variation)
    ElasticTransform(alpha=10, sigma=4, border_mode=cv2.BORDER_REFLECT, p=0.2),
    OpticalDistortion(distort_limit=0.08, shift_limit=0.08, 
                      border_mode=cv2.BORDER_REFLECT, p=0.2),
    GridDistortion(num_steps=4, distort_limit=0.08, 
                   border_mode=cv2.BORDER_REFLECT, p=0.2),
    
    # Brightness & Contrast variations
    RandomBrightnessContrast(brightness_limit=0.06, contrast_limit=0.06, p=0.5),
    RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.12, p=0.3),
    RandomBrightnessContrast(brightness_limit=0.18, contrast_limit=0.15, p=0.2),
    RandomGamma(gamma_limit=(92, 108), p=0.35),
    CLAHE(clip_limit=2.0, p=0.25),
    
    # Color variations (natural hue/saturation shifts)
    HueSaturationValue(hue_shift_limit=4, sat_shift_limit=8, val_shift_limit=8, p=0.45),
    HueSaturationValue(hue_shift_limit=7, sat_shift_limit=12, val_shift_limit=12, p=0.35),
    HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=15, p=0.25),
    RGBShift(r_shift_limit=8, g_shift_limit=8, b_shift_limit=8, p=0.35),
    RGBShift(r_shift_limit=12, g_shift_limit=12, b_shift_limit=12, p=0.25),
    ColorJitter(brightness=0.08, contrast=0.08, saturation=0.10, hue=0.04, p=0.3),
    
    # Sharpening & Emboss (texture changes)
    Sharpen(alpha=(0.15, 0.25), lightness=(0.85, 1.0), p=0.25),
    Sharpen(alpha=(0.05, 0.15), lightness=(0.9, 1.0), p=0.2),
    Emboss(alpha=(0.15, 0.25), strength=(0.3, 0.5), p=0.15),
    
    # Subtle environmental shadows (natural lighting variation)
    RandomShadow(shadow_roi=(0, 0, 1, 1), num_shadows_lower=1, num_shadows_upper=2, 
                 shadow_dimension=4, p=0.15),
])

# Process each image
img_files = [f for f in os.listdir(src_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

print(f"🌿 Processing {len(img_files)} images with {N_per_image} augmentations each...")
print(f"📁 Output directory: {out_dir}\n")

for fname in tqdm(img_files, desc="Augmenting images"):
    img_path = os.path.join(src_dir, fname)
    img = cv2.imread(img_path)
    if img is None:
        print(f"⚠️ Skipping {fname} (could not read)")
        continue

    base, ext = os.path.splitext(fname)
    
    # Save original resized version
    resized = cv2.resize(img, (256, 256))
    cv2.imwrite(os.path.join(out_dir, f"{base}_orig{ext}"), resized)

    # Generate augmented versions
    for i in range(N_per_image):
        augmented = aug(image=img)['image']
        out_name = f"{base}_aug{i:03d}{ext}"
        cv2.imwrite(os.path.join(out_dir, out_name), augmented)

total_images = len(img_files) * (N_per_image + 1)
print(f"\n✅ Augmentation complete!")
print(f"📊 Total images generated: {total_images}")
print(f"📂 Saved to: {out_dir}")

C:\Users\vansh\AppData\Local\Temp\ipykernel_20140\2376904909.py:37: UserWarning: Argument(s) 'shift_limit' are not valid for transform OpticalDistortion
  OpticalDistortion(distort_limit=0.08, shift_limit=0.08,
C:\Users\vansh\AppData\Local\Temp\ipykernel_20140\2376904909.py:63: UserWarning: Argument(s) 'num_shadows_lower, num_shadows_upper' are not valid for transform RandomShadow
  RandomShadow(shadow_roi=(0, 0, 1, 1), num_shadows_lower=1, num_shadows_upper=2,


🌿 Processing 39 images with 25 augmentations each...
📁 Output directory: Plum_Rust_Fungus_Augmented



Augmenting images:   0%|          | 0/39 [00:00<?, ?it/s]

Augmenting images: 100%|██████████| 39/39 [00:25<00:00,  1.52it/s]


✅ Augmentation complete!
📊 Total images generated: 1014
📂 Saved to: Plum_Rust_Fungus_Augmented
